# Hello Spark

Verbinden met het lokale Spark-cluster via **Spark Connect** (poort 15002), het protocol dat ook Databricks Connect gebruikt.

Vereist: `tilt up` draait en `pf-connect` is groen in de Tilt-UI.

**Catalog-isolatie:** Spark Connect en Spark Thrift draaien als gescheiden Spark-applicaties met elk een eigen in-memory catalog. Tabellen die `dbt-smoke` aanmaakt via Thrift zie je hier dus niet — voor gedeelde state heb je een Hive Metastore nodig (zie v2 in `../README.md`).

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .remote("sc://localhost:15002")
    .getOrCreate()
)
print("Spark version:", spark.version)

Spark version: 4.0.3


In [ ]:
# Eigen DataFrame opbouwen — dit landt op het cluster, niet op je laptop
df = spark.createDataFrame(
    [
        (1, "alice", 100.00, "2026-01-01"),
        (2, "bob",   250.00, "2026-01-02"),
        (3, "alice",  75.50, "2026-01-03"),
        (4, "carol", 500.00, "2026-01-03"),
        (5, "bob",   150.00, "2026-01-04"),
        (6, "alice", 300.00, "2026-01-05"),
        (7, "david", 425.75, "2026-01-05"),
        (8, "david", -900.20, "2026-01-06"),
        (9, "william", -300.40, "2036-01-06")
    ],
    ["order_id", "customer_id", "amount", "order_date"],
)
df.createOrReplaceTempView("orders")
df.show()

+--------+-----------+------+----------+
|order_id|customer_id|amount|order_date|
+--------+-----------+------+----------+
|       1|      alice| 100.0|2026-01-01|
|       2|        bob| 250.0|2026-01-02|
|       3|      alice|  75.5|2026-01-03|
|       4|      carol| 500.0|2026-01-03|
|       5|        bob| 150.0|2026-01-04|
|       6|      alice| 300.0|2026-01-05|
|       7|      david|425.75|2026-01-05|
|       8|      david|-900.2|2026-01-06|
|       9|    william|-300.4|2036-01-06|
+--------+-----------+------+----------+



In [ ]:
# exercise 10
from pyspark.sql import Row
products = [
    Row(name="Apple", price=1.5),
    Row(name="Laptop", price=999.0),
    Row(name="Coffee", price=4.5),
    Row(name="Monitor", price=299.0),
]
df = spark.createDataFrame(products)
df.filter(df.price > 50).show()


+-------+-----+
|   name|price|
+-------+-----+
| Laptop|999.0|
|Monitor|299.0|
+-------+-----+



In [4]:
# Spark SQL — analyse op het cluster, resultaat naar je notebook
spark.sql("""
    SELECT customer_id,
           COUNT(*)    AS n_orders,
           SUM(amount) AS lifetime_value
    FROM orders
    GROUP BY customer_id
    ORDER BY lifetime_value DESC
""").show()

+-----------+--------+-------------------+
|customer_id|n_orders|     lifetime_value|
+-----------+--------+-------------------+
|      carol|       1|              500.0|
|      alice|       3|              475.5|
|        bob|       2|              400.0|
|    william|       1|             -300.4|
|      david|       2|-474.45000000000005|
+-----------+--------+-------------------+



In [5]:
# DataFrame-API i.p.v. SQL — zelfde resultaat
from pyspark.sql import functions as F
(
    spark.table("orders")
    .groupBy("customer_id")
    .agg(F.count("*").alias("n_orders"), F.sum("amount").alias("lifetime_value"))
    .orderBy(F.col("lifetime_value").desc())
    .show()
)

+-----------+--------+-------------------+
|customer_id|n_orders|     lifetime_value|
+-----------+--------+-------------------+
|      carol|       1|              500.0|
|      alice|       3|              475.5|
|        bob|       2|              400.0|
|    william|       1|             -300.4|
|      david|       2|-474.45000000000005|
+-----------+--------+-------------------+



In [6]:
# Resultaat als pandas DataFrame voor verdere analyse / plotten
pdf = (
    spark.table("orders")
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
    .orderBy("order_date")
    .toPandas()
)
pdf

,order_date,revenue
0,2026-01-01,100.00
1,2026-01-02,250.00
2,2026-01-03,575.50
3,2026-01-04,150.00
4,2026-01-05,725.75
5,2026-01-06,-900.20
6,2036-01-06,-300.40
